# Vision Transformer in Pure NumPy 🔥
NumPy port of `vision_transformers.ipynb`: same sections (implement → data → utils → train → visualize), torch-free (CPU; `numpy`/`pillow`/`matplotlib` only).

| This notebook | NumPy module | Original |
|---|---|---|
| ViT Implementation | `vit_numpy.py` (`ViTForClassification`, `CrossEntropyLoss`, `AdamW`) | `vit.py` classes (were inlined) |
| Prepare Data | `data_numpy.py` (`prepare_data`) | `data.py` |
| Utils | `utils_numpy.py` | `utils.py` |
| Train ViT | `train_numpy.py` (`Trainer`) | `train.py` + inline `Trainer` |
| Visualize / Plot / Attention | `utils_numpy.py` | `inspect.ipynb` cells |

Cells import from the tested local modules, so notebook behavior matches `train_numpy.py` exactly. Images are `(B, C, H, W)`, sequences `(B, S, D)`; checkpoints are `.npz` (not `.pt`).

In [ ]:
#@title ViT Implementation 🔥 (pure NumPy; cf. vit.py / vit_numpy.py)
import math
import numpy as np
from vit_numpy import (
    NewGELUActivation, PatchEmbeddings, Embeddings, AttentionHead,
    MultiHeadAttention, FasterMultiHeadAttention, MLP, Block, Encoder,
    ViTForClassification, CrossEntropyLoss, AdamW,
)

config = {
    "patch_size": 4,  # Input image size: 32x32 -> 8x8 = 64 patches
    "hidden_size": 48,
    "num_hidden_layers": 4,
    "num_attention_heads": 4,
    "intermediate_size": 4 * 48,  # 4 * hidden_size
    "hidden_dropout_prob": 0.0,
    "attention_probs_dropout_prob": 0.0,
    "initializer_range": 0.02,
    "image_size": 32,
    "num_classes": 10,  # num_classes of CIFAR10
    "num_channels": 3,
    "qkv_bias": True,
    "use_faster_attention": True,
}
# Same guards as train.py / train_numpy.py
assert config["hidden_size"] % config["num_attention_heads"] == 0
assert config["intermediate_size"] == 4 * config["hidden_size"]
assert config["image_size"] % config["patch_size"] == 0

# Shape demo: (B, C, H, W) images -> (B, K) logits; attentions are per-layer (B, H, S, S)
model = ViTForClassification(config, rng=np.random.default_rng(0))
print(f"learnable tensors: {len(model.named_parameters())}")
x = np.random.default_rng(0).standard_normal((2, 3, 32, 32)).astype(np.float32)
logits, _ = model.forward(x, training=False)
print(f"images {x.shape} -> logits {logits.shape}")
logits, attns = model.forward(x, output_attentions=True, training=False)
print(f"layers: {len(attns)}, maps per layer: {attns[0].shape} (B, heads, S, S)")

In [ ]:
#@title Prepare Data 📊 (cf. data.py / data_numpy.py)
from data_numpy import prepare_data

import os
_CANDIDATES = [
    os.environ.get("VIT_DATA_ROOT", ""),
    "/Users/jinghuayao/Downloads/vision-transformer-from-scratch/data",
    "./data",
    "../vision-transformer-from-scratch/data",
]
DATA_ROOT = next(
    (c for c in _CANDIDATES
     if c and os.path.isdir(os.path.join(c, "cifar-10-batches-py"))),
    None,
)
assert DATA_ROOT is not None, "CIFAR-10 pickles not found; set VIT_DATA_ROOT"
print("DATA_ROOT =", DATA_ROOT)
trainloader, testloader, classes = prepare_data(
    batch_size=4, train_sample_size=8, test_sample_size=8,
    data_root=DATA_ROOT, seed=0)
images, labels = next(iter(trainloader))
print(f"batch images {images.shape} dtype={images.dtype} "
      f"range=[{images.min():.2f}, {images.max():.2f}]  (normalized to [-1, 1])")
print(f"batch labels {labels.shape}: {labels.tolist()}")
print("classes:", classes)

In [ ]:
#@title Utils 🛠️ (cf. utils.py / utils_numpy.py)
from utils_numpy import (
    save_experiment, save_checkpoint, load_experiment,
    plot_metrics, visualize_images, visualize_attention,
)
print("utils ready; checkpoints are saved as .npz")

In [ ]:
#@title Train ViT 🧠 🏋🏽 (cf. train.py; Trainer imported from train_numpy.py)
exp_name = 'vit-numpy-demo'  #@param {type:"string"}
batch_size = 32  #@param {type: "integer"}
epochs = 2  #@param {type: "integer"}
lr = 1e-2  #@param {type: "number"}
save_model_every = 0  #@param {type: "integer"}

from train_numpy import Trainer

# Small demo subset so the notebook runs fast on CPU; raise epochs/samples for a full run
trainloader, testloader, _ = prepare_data(
    batch_size=batch_size, train_sample_size=256, test_sample_size=64,
    data_root=DATA_ROOT, seed=0)
model = ViTForClassification(config, rng=np.random.default_rng(0))
optimizer = AdamW(model, lr=lr, weight_decay=1e-2)
loss_fn = CrossEntropyLoss()
trainer = Trainer(model, optimizer, loss_fn, exp_name)
trainer.train(trainloader, testloader, epochs,
            save_model_every_n_epochs=save_model_every)

In [ ]:
#@title Visualize Dataset
# Show some training images (saved to samples.png and displayed)
from IPython.display import Image, display
visualize_images(data_root=DATA_ROOT, output="samples.png")
display(Image("samples.png"))

In [ ]:
#@title Plot training Results
config, model, train_losses, test_losses, accuracies = load_experiment(exp_name)
print("train_losses:", [round(v, 4) for v in train_losses])
print("test_losses:", [round(v, 4) for v in test_losses])
print("accuracies:", [round(v, 4) for v in accuracies])
plot_metrics(train_losses, test_losses, accuracies, output="metrics.png")
display(Image("metrics.png"))

In [ ]:
#@title Visualize Attention
visualize_attention(model, data_root=DATA_ROOT, output="attention.png")
display(Image("attention.png"))